In [0]:
%pip install cloudscraper

In [0]:
import requests
import json
from bs4 import BeautifulSoup
import re
import pandas as pd
import time

In [0]:
"""
Reusable Web Scraper for Wheree.com and Similar Structured Sites
=================================================================

REQUIREMENTS:
-------------
- cloudscraper: pip install cloudscraper
- pandas: pip install pandas
- beautifulsoup4: pip install beautifulsoup4

"""

# ============================================================================
# USER CONFIGURATION - Modify these variables for your scraping needs
# ============================================================================

# The base URL to scrape (paste your target URL here)
BASE_URL = "https://www.wheree.com/get-brands?page=1&category_id=82&category_slug=Convenience_Stores&location_id=231&location_level=0&location_slug=United_States"

# Page range to scrape (start_page, end_page)
# Example: (1, 5) scrapes pages 1 through 4
# For all 1926 pages, use: (1, 1927)
START_PAGE = 1
END_PAGE = 5

# Output filename for the scraped data
OUTPUT_FILENAME = "convenience stores.csv"

# Retry settings for handling network issues
MAX_RETRIES = 5
REQUEST_TIMEOUT = 60

# ============================================================================
# IMPORTS AND SETUP
# ============================================================================

import cloudscraper
import pandas as pd
import time
import re
from bs4 import BeautifulSoup

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def extract_maprows_info(parsed):
    """
    Parse and normalize data from various response formats.
    
    This function is tolerant to multiple payload structures:
    - JSON dict with 'mapRows' -> {'data': [...]}
    - JSON dict with 'brands' containing HTML
    - Raw HTML containing brand blocks with data-brand attributes
    
    Args:
        parsed: Either a dict (from JSON response) or string (HTML response)
    
    Returns:
        list: List of normalized record dictionaries, or [] on failure
    """
    records = []
    
    try:
        # Case 1: JSON dict with mapRows.data structure
        if isinstance(parsed, dict):
            # Try to extract from nested mapRows.data
            mr = parsed.get("mapRows")
            if isinstance(mr, dict):
                data = mr.get("data")
                if isinstance(data, list) and data:
                    return data
            
            # Sometimes data is at the top level
            if isinstance(parsed.get("data"), list):
                return parsed.get("data")
            
            # Try HTML embedded in 'brands' key
            brands_html = parsed.get("brands")
            if isinstance(brands_html, str):
                parsed = brands_html  # Fall through to HTML parsing

        # Case 2: HTML string - extract using BeautifulSoup
        if isinstance(parsed, str):
            soup = BeautifulSoup(parsed, "html.parser")
            
            # Find all business listing containers
            # Common classes: category__top__item, category__item, category-item
            items = soup.find_all("div", class_=re.compile(r"category__top__item|category__item|category-item"))
            
            # Fallback: find any element with data-brand attribute
            if not items:
                items = soup.find_all(attrs={"data-brand": True})
            
            # Extract data from each listing
            for div in items:
                rec = {}
                
                # Extract ID
                rec["id"] = div.get("data-brand") or div.get("data-id") or None
                
                # Extract URL and name
                a = div.find("a")
                if a:
                    rec["url"] = a.get("href")
                    # Try to find a heading element for the name
                    title = a.find(["h1", "h2", "h3", "h4", "h5"])
                    if title:
                        rec["name"] = title.get_text(strip=True)
                    else:
                        # Fallback to all text content
                        rec["name"] = a.get_text(separator=" ", strip=True)
                else:
                    rec["url"] = None
                    rec["name"] = div.get_text(separator=" ", strip=True)
                
                # Extract image
                img = div.find("img")
                if img:
                    rec["image"] = img.get("src") or img.get("data-src")
                    if not rec.get("name"):
                        rec["name"] = img.get("alt")
                
                # Store raw HTML snippet for debugging
                rec["raw_html"] = str(div)
                
                records.append(rec)
                
            return records
            
    except Exception as e:
        print(f"Error parsing data: {e}")
    
    return records


def extract_ids_and_names(detail_str):
    """
    Safely extract id and name pairs from string representation of list.
    
    Args:
        detail_str: String representation of a list of dicts
    
    Returns:
        list: List of (id, name) tuples, or None on failure
    """
    import ast
    try:
        data_list = ast.literal_eval(detail_str)
        return [(item['id'], item['name']) for item in data_list]
    except Exception:
        return None


def extract_first_name(detail_list):
    """
    Extract the 'name' field from the first item in a list of dicts.
    
    Args:
        detail_list: List of dictionaries
    
    Returns:
        str: Name from first item, or None if not found
    """
    try:
        if isinstance(detail_list, list) and len(detail_list) > 0:
            first_item = detail_list[0]
            if isinstance(first_item, dict):
                return first_item.get('name')
    except Exception:
        pass
    return None


def extract_state(address):
    """
    Extract US state code (2-letter) from address string.
    
    Uses regex to find pattern: 2 uppercase letters followed by 5-digit ZIP
    
    Args:
        address: Full address string
    
    Returns:
        str: 2-letter state code, or None if not found
    """
    if pd.isna(address):
        return None
    
    state_pattern = re.compile(r'\b([A-Z]{2})\b(?=\s*\d{5})')
    match = state_pattern.search(address)
    
    if match:
        return match.group(1)
    return None


def clean_dataframe(df):
    """
    Clean and normalize the scraped dataframe.
    
    Performs the following operations:
    - Extracts IDs and names from 'sub_cat' column
    - Extracts state codes from addresses
    - Drops unnecessary columns
    - Renames columns for clarity
    - Converts list columns to comma-separated strings
    
    Args:
        df: Raw pandas DataFrame
    
    Returns:
        DataFrame: Cleaned and normalized DataFrame
    """
    # Extract category information from sub_cat column
    if 'sub_cat' in df.columns:
        df['extracted'] = df['sub_cat'].apply(extract_ids_and_names)
        df['ids'] = df['extracted'].apply(lambda x: [i[0] for i in x] if x else None)
        df['names'] = df['sub_cat'].apply(
            lambda x: [item['name'] for item in x] if isinstance(x, list) else None
        )
        df['category'] = df['sub_cat'].apply(extract_first_name)
    
    # Extract state from full address
    if 'fulladdress' in df.columns:
        df['state'] = df['fulladdress'].apply(extract_state)
    
    # Drop unnecessary columns
    drop_cols = [
        'alias', 'ranking', 'rating', 'city', 'number_of_rates', 
        'price_level', 'created_at', 'category_id', 'category2_id', 
        'category3_id', 'short_description', 'image', 'openTime', 
        'sub_cat', 'affiliate', 'extracted', 'ids'
    ]
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True, errors='ignore')
    
    # Rename columns for clarity
    df.rename(columns={
        'fulladdress': 'address',
        'names': 'categories',
        'category': 'sub_category',
    }, inplace=True)
    
    # Convert list columns to comma-separated strings
    if 'categories' in df.columns:
        df['categories'] = df['categories'].apply(
            lambda x: ', '.join(x) if isinstance(x, list) else x
        )
    
    return df


# ============================================================================
# MAIN SCRAPING LOGIC
# ============================================================================

def detect_total_pages(scraper, base_url):
    """
    Attempt to detect the total number of pages available.
    
    This function fetches the first page and tries to extract pagination info
    from various common structures in the response.
    
    Args:
        scraper: CloudScraper instance
        base_url: The base URL to check
    
    Returns:
        int: Total number of pages, or None if unable to detect
    """
    try:
        # Fetch the first page
        if re.search(r"page=\d+", base_url):
            first_page_url = re.sub(r"page=\d+", "page=1", base_url)
        else:
            separator = "&" if "?" in base_url else "?"
            first_page_url = f"{base_url}{separator}page=1"
        
        resp = scraper.get(first_page_url, timeout=30)
        resp.raise_for_status()
        
        # Try parsing as JSON first
        try:
            data = resp.json()
            
            # Common JSON structures for pagination
            # Check for total_pages, last_page, totalPages, etc.
            possible_keys = [
                'total_pages', 'totalPages', 'last_page', 'lastPage',
                'page_count', 'pageCount', 'pages', 'total'
            ]
            
            for key in possible_keys:
                if key in data:
                    return int(data[key])
            
            # Check nested structures like pagination.total_pages
            if 'pagination' in data and isinstance(data['pagination'], dict):
                for key in possible_keys:
                    if key in data['pagination']:
                        return int(data['pagination'][key])
            
            # Check if mapRows has pagination info
            if 'mapRows' in data and isinstance(data['mapRows'], dict):
                for key in possible_keys:
                    if key in data['mapRows']:
                        return int(data['mapRows'][key])
        
        except Exception:
            # If JSON parsing fails, try HTML parsing
            soup = BeautifulSoup(resp.text, "html.parser")
            
            # Look for pagination elements
            pagination = soup.find('div', class_=re.compile(r'pagination|pager'))
            if pagination:
                # Find all page links
                page_links = pagination.find_all('a', href=re.compile(r'page=\d+'))
                if page_links:
                    max_page = 0
                    for link in page_links:
                        match = re.search(r'page=(\d+)', link.get('href', ''))
                        if match:
                            max_page = max(max_page, int(match.group(1)))
                    if max_page > 0:
                        return max_page
    
    except Exception as e:
        print(f"Could not auto-detect total pages: {e}")
    
    return None


def main():
    """
    Main function to orchestrate the web scraping process.
    
    Steps:
    1. Initialize scraper with Cloudflare bypass
    2. Detect total available pages
    3. Get user confirmation on page range
    4. Loop through specified page range
    5. Fetch and parse each page
    6. Extract business data
    7. Clean and normalize data
    8. Save to CSV file
    """
    global START_PAGE, END_PAGE, OUTPUT_FILENAME, MAX_RETRIES, REQUEST_TIMEOUT
    
    print("=" * 70)
    print("Web Scraper Starting")
    print("=" * 70)
    print(f"Target URL: {BASE_URL}")
    
    # Initialize cloudscraper (handles Cloudflare challenges automatically)
    try:
        scraper = cloudscraper.create_scraper()
    except Exception as e:
        raise RuntimeError(f"Failed to create scraper: {e}")
    
    # Attempt to detect total pages available
    print("\nDetecting total pages available...")
    total_pages = detect_total_pages(scraper, BASE_URL)
    
    if total_pages:
        print(f"✓ Detected {total_pages} total pages available")
    else:
        print("✗ Could not auto-detect total pages")
        print("  (The site may not expose this information)")
    
    # Show configured range
    print(f"\nConfigured to scrape: Pages {START_PAGE} to {END_PAGE - 1}")
    
    if total_pages and END_PAGE > total_pages + 1:
        print(f"⚠ WARNING: Your END_PAGE ({END_PAGE}) exceeds available pages ({total_pages})")
        print(f"  Recommendation: Set END_PAGE to {total_pages + 1}")
    
    print(f"Output file: {OUTPUT_FILENAME}")
    print("=" * 70)
    
    # Ask user to confirm or modify range
    if total_pages:
        print(f"\n📊 There are {total_pages} pages available.")
        print(f"   You're configured to scrape pages {START_PAGE} to {END_PAGE - 1} ({END_PAGE - START_PAGE} pages)")
        user_input = input("\nWould you like to continue or enter a different end page? (continue/number): ").strip().lower()
        
        if user_input.isdigit():
            new_end = int(user_input) + 1  # +1 because range is exclusive
            if new_end <= START_PAGE:
                print(f"Error: End page must be greater than start page ({START_PAGE})")
                return
            if new_end > total_pages + 1:
                print(f"Warning: End page {new_end - 1} exceeds available pages ({total_pages})")
                confirm = input("Continue anyway? (yes/no): ").strip().lower()
                if confirm not in ['yes', 'y']:
                    print("Scraping cancelled by user.")
                    return
            END_PAGE = new_end
            print(f"✓ Updated to scrape pages {START_PAGE} to {END_PAGE - 1}")
        elif user_input not in ['continue', 'c', 'yes', 'y']:
            print("Scraping cancelled by user.")
            return
    else:
        print(f"\n⚠ Could not detect total pages")
        print(f"   Configured to scrape pages {START_PAGE} to {END_PAGE - 1}")
        user_input = input("\nContinue with configured range? (yes/no): ").strip().lower()
        
        if user_input not in ['yes', 'y']:
            print("Scraping cancelled by user.")
            return
    
    # Storage for all scraped records
    all_records = []
    
    # Loop through each page
    pages = list(range(START_PAGE, END_PAGE))
    
    for page_num in pages:
        # Build the page-specific URL
        if re.search(r"page=\d+", BASE_URL):
            # Replace existing page parameter
            page_url = re.sub(r"page=\d+", f"page={page_num}", BASE_URL)
        else:
            # Append page parameter
            separator = "&" if "?" in BASE_URL else "?"
            page_url = f"{BASE_URL}{separator}page={page_num}"
        
        # Fetch page with retry logic
        resp = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                resp = scraper.get(page_url, timeout=REQUEST_TIMEOUT)
                resp.raise_for_status()
                break
            except Exception as e:
                if attempt < MAX_RETRIES:
                    wait_time = 2 * attempt  # Exponential backoff
                    time.sleep(wait_time)
        
        # Skip if page fetch failed
        if resp is None:
            continue
        
        # Parse response (try JSON first, fall back to HTML)
        parsed = None
        try:
            parsed = resp.json()
        except Exception:
            parsed = resp.text
        
        # Extract records from parsed data
        try:
            records = extract_maprows_info(parsed)
        except Exception:
            records = []
        
        # Update progress
        if records:
            all_records.extend(records)
    
    # Process and save results
    print("\n" + "=" * 70)
    if not all_records:
        print("WARNING: No records were collected from any page")
        print("This could mean:")
        print("  - The URL structure doesn't match the expected format")
        print("  - The site requires authentication")
        print("  - The site structure has changed")
        print("  - Network issues prevented successful fetching")
        return
    
    print(f"SUCCESS: Collected {len(all_records)} total records")
    print("=" * 70)
    
    # Convert to DataFrame
    df_all = pd.DataFrame(all_records)
    
    print("\nCleaning data...")
    # Clean and normalize the data
    df_all = clean_dataframe(df_all)
    
    # Save to CSV
    df_all.to_csv(OUTPUT_FILENAME, index=False)
    print(f"\nSaved {len(df_all)} records to '{OUTPUT_FILENAME}'")
    
    # Display preview
    print("\n" + "=" * 70)
    print("Preview of scraped data (first 20 rows):")
    print("=" * 70)
    print(df_all.head(20).to_string())
    
    print("\n" + "=" * 70)
    print(f"Column names: {list(df_all.columns)}")
    print(f"Total rows: {len(df_all)}")
    print("=" * 70)


# ============================================================================
# SCRIPT EXECUTION
# ============================================================================

if __name__ == "__main__":
    main()